# 🛰️ Punjab GeoGuardian - AI-Based Monitoring of Urban Expansion and Farmland Loss
**AIRI Team PITB - AI Internship Task 3** · End-to-End GeoAI and Satellite Intelligence Project
**Prepared by:** Hamza Nadeem · **Submitted to:** Omar Farooq - Associate AI/ML Engineer

This notebook runs the complete GeoAI workflow with **Google Earth Engine + Sentinel-2** and Python:

`Select AOI → Load Sentinel-2 → Cloud-free composite → NDVI & NDBI → Classify land cover → Compare 2019 vs 2025 → Detect changed pixels → Calculate area → Maps & dashboard → Validation & error analysis → Report`

| Notebook part | PDF requirement |
|---|---|
| 1-2 Setup & AOI | §5 AOI selection, §11 data collection |
| 3 Data collection | §11-12 Sentinel-2, date/cloud filter, median composite, clip |
| 4 Pre-processing maps | §12 RGB / NDVI / NDBI for both years (Week-1 deliverables) |
| 5 Training samples | §14 ≥ 50 samples per class, spread over the AOI |
| 6 Classification | §13 Option A (rules) + Option B (Random Forest / XGBoost) |
| 7 Change detection | §15 agri-to-urban conversion, area table, hotspots |
| 8 Validation | §18 ≥ 50 manually checked points, accuracy |
| 9 Error analysis | §19 ≥ 15 cases + improvements |
| 10 Maps | §16 all required maps (≥ 8) |
| 11 Interactive map / dashboard | §17 Folium layer toggles + Streamlit |
| 12 Exports, report, LinkedIn | §21 deliverables, §25 portfolio section |

> ⚠️ Learning prototype - **not** an official government monitoring system. Results are *possible / estimated* changes.

**Runtime:** about 15-25 min for the first run (most of it is the Earth Engine download). Re-runs take about 2 min.

## 1. Setup

In [6]:
# --- Settings you may change -------------------------------------------------------
EE_PROJECT   = "hamza-geoguardian"   # <-- your Google Cloud project registered for Earth Engine
USE_DRIVE    = True                    # keep data and outputs on Google Drive (= Drive submission)
PROJECT_DIR  = "/content/drive/MyDrive/punjab-geoguardian"   # where the project folder lives
REPO_URL     = ""                      # optional: your GitHub repo URL (used if PROJECT_DIR is missing)
CONFIG_FILE  = "config/lahore_raiwind.json"
USE_SYNTHETIC = False                  # True = offline smoke test only (watermarked, not real results)

In [7]:
import os, sys, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB and USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
if not os.path.exists(PROJECT_DIR):
    if REPO_URL:
        subprocess.run(["git", "clone", REPO_URL, PROJECT_DIR], check=True)
    else:
        raise FileNotFoundError(f"Upload the project folder to {PROJECT_DIR} (or set REPO_URL).")
os.chdir(PROJECT_DIR)
sys.path.insert(0, PROJECT_DIR)
if IN_COLAB:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
print("Working directory:", os.getcwd())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Working directory: /content/drive/MyDrive/punjab-geoguardian


## 2. Area of Interest and configuration

In [8]:
import json, pandas as pd
from IPython.display import Image, display, Markdown, HTML
from geoguardian.config import ProjectConfig
from geoguardian.pipeline import GeoGuardian

if USE_SYNTHETIC:
    from geoguardian.synthetic import make_synthetic_stack
    cfg = ProjectConfig(aoi_key="synthetic_demo", aoi_name="Synthetic test scene", synthetic=True,
                        data_dir="data_synthetic", output_dir="outputs_synthetic")
else:
    cfg = ProjectConfig.from_json(CONFIG_FILE)
cfg.validate()
pd.DataFrame({
    "Setting": ["Author", "AOI", "AOI mode", "Centre (lat, lon)", "Size", "Years compared",
                "Extra years (trend)", "Season (MM-DD)", "Cloud filter", "Satellite"],
    "Value": [cfg.author_name, cfg.aoi_name, cfg.aoi_mode, f"{cfg.center_lat}, {cfg.center_lon}",
              f"{cfg.width_km} x {cfg.height_km} km", f"{cfg.baseline_year} and {cfg.recent_year}",
              cfg.extra_years, f"{cfg.season_start} to {cfg.season_end}",
              f"< {cfg.max_cloud_pct}% scene cloud + Cloud Score+ pixel mask",
              "Sentinel-2 L2A harmonized (10 m)"]})

,Setting,Value
0,Author,Hamza Nadeem
1,AOI,Lahore peri-urban expansion zone near Raiwind ...
2,AOI mode,rectangle
3,"Centre (lat, lon)","31.4, 74.25"
4,Size,15.0 x 15.0 km
5,Years compared,2019 and 2025
6,Extra years (trend),[2022]
7,Season (MM-DD),02-01 to 03-31
8,Cloud filter,< 20.0% scene cloud + Cloud Score+ pixel mask
9,Satellite,Sentinel-2 L2A harmonized (10 m)


**Why this AOI?** The Raiwind Road corridor in south-western Lahore is a well-known peri-urban growth belt (the PDF's own example). Farmland, brick kilns and new housing schemes sit side by side there. To analyse another city or a full district, change `aoi_key`, `center_lat/lon` or `aoi_mode="district"` + `district_name` in the config file (presets: `geoguardian.config.AOI_PRESETS`).

In [9]:
if not USE_SYNTHETIC:
    from geoguardian.gee_data import init_earth_engine, preview_map
    init_earth_engine(EE_PROJECT)
    m = preview_map(cfg)   # live Earth Engine layers with toggles + Google satellite basemap
    display(m)             # 📸 Week-1 deliverable: screenshot the AOI boundary from this map

[gee] Earth Engine initialised (project=hamza-geoguardian)


Map(center=[31.400031574400867, 74.24999842954013], controls=(WidgetControl(options=['position', 'transparent_…

## 3. Data collection (Sentinel-2 via Google Earth Engine)
Cloud-masked **median composites** for each year (same Feb-Mar season), clipped to the AOI and downloaded as one GeoTIFF stack on a 10 m UTM grid.

In [10]:
FORCE_DOWNLOAD = False
if USE_SYNTHETIC:
    make_synthetic_stack(cfg)
elif FORCE_DOWNLOAD or not cfg.stack_path.exists():
    from geoguardian.gee_data import collect_data
    collect_data(cfg)
meta = json.loads(cfg.stack_meta_path.read_text())
pd.DataFrame([{ "Year": y, "Date range": f"{i['start']} to {i['end']}", "Image dates": i["n_dates"],
                "Scenes": i["n_scenes"], "Tiles": ", ".join(sorted(set(i["tiles"]))),
                "Window widened (days)": i["window_extended_days"]}
              for y, i in meta["image_info"].items()])

,Year,Date range,Image dates,Scenes,Tiles,Window widened (days)
0,2019,2019-02-01 to 2019-03-31,11,11,43RDQ,0
1,2022,2022-02-01 to 2022-03-31,18,19,43RDQ,0
2,2025,2025-02-01 to 2025-03-31,13,14,43RDQ,0


## 4. Pre-processing outputs - RGB, NDVI, NDBI (Week-1 deliverables)

In [11]:
project = GeoGuardian(cfg)
project.load().preprocessing_maps()
F = project.figures
for k in ["aoi", "rgb_pair", "ndvi_b", "ndvi_r", "ndbi_b", "ndbi_r"]:
    display(Image(filename=str(F[k]), width=900 if k in ("aoi", "rgb_pair") else 560))

Output hidden; open in https://colab.research.google.com to view.

**Reading the indices**
- **NDVI = (NIR − Red)/(NIR + Red)**: green crops and trees are high (≈0.4-0.9); built-up and bare soil are low (≈0.1-0.2); water is negative.
- **NDBI = (SWIR − NIR)/(SWIR + NIR)**: built-up surfaces are high, vegetation is low. Dry bare soil is *also* high, so NDBI alone confuses fallow fields with construction. That is why NDVI, NDBI, MNDWI and all bands are combined in a classifier.

## 5. Training samples (≥ 50 per class)

In [12]:
project.build_training()
s = project.samples
display(s.groupby("class_name").agg(samples=("sample_id", "count"), spatial_blocks=("block", "nunique")))
display(s.groupby(["class_name", "source"]).size().rename("samples").to_frame())
print("Visual spot-check sheet:", project.p.validation / "training_samples_review.pdf",
      "(created in step 8). To drop a bad sample, set keep=False in",
      project.p.validation / "training_samples_review.csv", "and re-run.")

[samples] training locations per class: {'Built-up': 200, 'Other': 200, 'Vegetation': 200} (x 3 years = 1800 training rows)


,samples,spatial_blocks
class_name,,
Built-up,200,25
Other,200,20
Vegetation,200,25


samples
class_name source                                                     
Built-up   T1: WorldCover = Dynamic World (all years) + sp...      200
Other      T1: WorldCover = Dynamic World (all years) + sp...      183
           T2: Dynamic World recent year + spectral purity          17
Vegetation T1: WorldCover = Dynamic World (all years) + sp...      200

Visual spot-check sheet: outputs/validation/training_samples_review.pdf (created in step 8). To drop a bad sample, set keep=False in outputs/validation/training_samples_review.csv and re-run.


## 6. Land classification - Option A (rules) vs Option B (Random Forest / XGBoost)

In [13]:
project.classify()
cv = pd.DataFrame({k: {m: v for m, v in d.items() if m != "confusion_matrix"}
                   for k, d in project.model.all_cv.items()}).T
display(cv)
print("Selected model:", project.model.name)
print("Tuned rule thresholds:", project.model.rule["tuned"])
display(project.model.importances.rename("importance").to_frame())

[classify] random_forest  spatial-CV  OA=0.917  macro-F1=0.917  kappa=0.876
[classify] xgboost        spatial-CV  OA=0.916  macro-F1=0.916  kappa=0.874
[classify] selected model: random_forest


,overall_accuracy,macro_f1,kappa,note
rule_based,0.6739,0.6445,0.5108,thresholds tuned on the same samples (optimistic)
random_forest,0.9172,0.9171,0.8758,NaN
xgboost,0.9161,0.916,0.8742,NaN


Selected model: random_forest
Tuned rule thresholds: {'ndvi_veg': 0.35, 'ndbi_built': 0.1, 'mndwi_water': -0.1}


,importance
NDVI,0.242669
B11,0.137927
B12,0.113107
B2,0.110636
NDBI,0.099516
B8,0.090928
B4,0.086028
MNDWI,0.069107
B3,0.050083


## 7. Change detection, area calculation and hotspots

In [14]:
from geoguardian.change import hotspot_table
project.detect_change()
display(project.change.stats)
display(Markdown(f"**Transition matrix {cfg.baseline_year} → {cfg.recent_year} (km²)**"))
display(project.change.transition)
display(project.change.timeseries)
display(hotspot_table(project.change, project.scene.grid, 10)[
    ["rank", "converted_km2", "converted_pct_of_cell", "location", "center_lat", "center_lon"]])

[change] conversion 12.31 km^2 (5.47% of AOI); candidates before filters 14.40 km^2


,Metric,Value
0,Total AOI Area,225.00 sq. km
1,Area analysed (cloud-free both years),225.00 sq. km
2,Vegetation / Agriculture Area in 2019,83.40 sq. km
3,Vegetation / Agriculture Area in 2025,73.09 sq. km
4,Built-up Area in 2019,112.82 sq. km
5,Built-up Area in 2025,118.42 sq. km
6,Net Built-up Growth,+5.60 sq. km
7,Agriculture-to-Urban Conversion (estimated),12.31 sq. km (1231 ha)
8,Percentage of AOI Converted,5.47 %
9,Share of 2019 Vegetation Converted,14.76 %


**Transition matrix 2019 → 2025 (km²)**

,Vegetation 2025,Built-up 2025,Other 2025
Vegetation 2019,57.495,14.402,11.499
Built-up 2019,11.350,93.017,8.452
Other 2019,4.245,11.005,13.536


,year,Vegetation_km2,Built-up_km2,Other_km2,Built-up_percent
0,2019,83.395,112.819,28.786,50.14
1,2022,92.799,108.869,23.333,48.39
2,2025,73.090,118.424,33.487,52.63


,rank,converted_km2,converted_pct_of_cell,location,center_lat,center_lon
0,1,0.4605,46.05,north-western part of the AOI (9.2 km from cen...,31.46275,74.18640
1,2,0.3576,35.76,north-western part of the AOI (8.5 km from cen...,31.45373,74.18648
2,3,0.3426,34.26,north-western part of the AOI (8.6 km from cen...,31.46282,74.19692
3,4,0.3081,30.81,north-western part of the AOI (9.9 km from cen...,31.46269,74.17587
4,5,0.2189,21.89,central part of the AOI (1.0 km from centre),31.39992,74.23953
5,6,0.2123,21.23,north-western part of the AOI (9.2 km from cen...,31.45367,74.17595
6,7,0.2078,20.78,north-western part of the AOI (8.6 km from cen...,31.44464,74.17603
7,8,0.2009,20.09,northern part of the AOI (5.4 km from centre),31.44497,74.22865
8,9,0.1889,18.89,northern part of the AOI (5.1 km from centre),31.44503,74.23917
9,10,0.1729,17.29,north-western part of the AOI (4.5 km from cen...,31.43595,74.22872


## 8. Validation - 60 manually checked points (PDF minimum 50)
1. The next cell draws 60 stratified points (15 vegetation, 15 built-up, 15 other, 15 detected conversion), all ≥ 100 m from training samples.
2. **Label them by eye** with the widget below. For each point, look at the 2019/2025 chips and click the Google Maps link for high-resolution imagery. Choose the class you see in 2025 and whether vegetation became built-up. The model's prediction is hidden, so your labels are unbiased.
3. Labels are saved to `outputs/validation/validation_points.csv` (you can also edit it in Google Sheets). Then run the cells after the widget.

In [15]:
project.validation_sheet()
from geoguardian.labeler import ValidationLabeler
ValidationLabeler(project.p.validation / "validation_points.csv", project.p.validation / "chips").show()

[validation] kept 60 existing manual labels from validation_points.csv


In [16]:
# Run this after labelling (it reloads the CSV). Unlabelled points are marked PROVISIONAL.
project.assess()
v = project.validation
print(f"Points: {v['n_points']} | manual: {v['n_manual']} | correct: {v['correct']} | incorrect: {v['incorrect']}")
print(f"Approximate accuracy: {100*v['overall_accuracy']:.1f}%  (area-weighted {100*v['area_weighted_overall_accuracy']:.1f}%), kappa {v['kappa']:.3f}")
print(f"Rule-based (Option A) on the same points: {100*project.rule_validation['overall_accuracy']:.1f}%")
display(v["per_class"]); display(v["confusion_matrix"])
display(pd.DataFrame([v["change"]]).T.rename(columns={0: "value"}))
display(v["table"])

[validation] 60 points, 29 correct -> accuracy 48.3% (area-weighted 56.6%), kappa 0.11
[validation] 60 points, 12 correct -> accuracy 20.0% (area-weighted 19.8%), kappa -0.04
[errors] 36 cases analysed (36 incorrect, 0 weak-correct)
Points: 60 | manual: 60 | correct: 29 | incorrect: 31
Approximate accuracy: 48.3%  (area-weighted 56.6%), kappa 0.114
Rule-based (Option A) on the same points: 20.0%


,class,reference_points,mapped_points,producers_accuracy,users_accuracy
0,Vegetation,12,15,0.500,0.400
1,Built-up,40,30,0.500,0.667
2,Other,8,15,0.375,0.200


Predicted,Vegetation,Built-up,Other
Actual,,,
Vegetation,6,5,1
Built-up,9,20,11
Other,0,5,3


,value
true_positive,7.000
false_positive (false conversion),8.000
false_negative (missed change),7.000
true_negative,38.000
conversion_precision_users_acc,0.467
conversion_recall_producers_acc,0.500
change_overall_accuracy,0.750
error_adjusted_conversion_km2,33.696
ci95_km2,22.777
ci95_low_km2,10.919


,Point ID,Latitude,Longitude,Actual Class,Predicted Class,Correct / Incorrect,Actual Change,Predicted Change,Notes,Label Source
0,1,31.382351,74.242830,Built-up,Built-up,Correct,yes,yes,new houses on former farmland new housing sche...,Manual visual check
1,2,31.414685,74.218479,Built-up,Built-up,Correct,no,yes,"old built-up area, no change existing houses s...",Manual visual check
2,3,31.465452,74.259214,Built-up,Built-up,Correct,no,no,"old built-up area, no change existing houses s...",Manual visual check
3,4,31.371223,74.253119,Other,Other,Correct,no,no,"bare/fallow field, no buildings cleared plot, ...",Manual visual check
4,5,31.415117,74.172396,Vegetation,Built-up,Incorrect,no,yes,"crop field, no change wheat field / green crop...",Manual visual check
5,6,31.375064,74.231216,Built-up,Other,Incorrect,yes,no,,Manual visual check
6,7,31.367961,74.235374,Other,Built-up,Incorrect,no,no,,Manual visual check
7,8,31.411232,74.325390,Vegetation,Other,Incorrect,no,no,,Manual visual check
8,9,31.387899,74.298212,Other,Built-up,Incorrect,no,yes,,Manual visual check
9,10,31.360089,74.187704,Built-up,Other,Incorrect,no,no,,Manual visual check


## 9. Error analysis (≥ 15 cases)

In [17]:
e = project.errors
display(e["table"])
display(Markdown("**Most common reasons:** " + ", ".join(f"{k} ({n})" for k, n in e["reason_counts"].items())))
display(Markdown("**Suggested improvements:**\n" + "\n".join(f"- {i}" for i in e["improvements"])))

,Case,Location / Point,Latitude,Longitude,Expected Result,Model Result,Error Type,Possible Reason (suggested),Evidence,Analyst Notes
0,1,Point 5,31.415117,74.172396,Vegetation (no change),Conversion detected,False conversion,Bare / fallow soil confused with built-up,"NDVI 2019=0.50, 2025=0.23; NDBI 2025=-0.02; cl...","crop field, no change wheat field / green crop..."
1,2,Point 6,31.375064,74.231216,Vegetation-to-built-up,Other (no change),Missed change,New road / construction site confused with bar...,"NDVI 2019=0.52, 2025=0.06; NDBI 2025=0.00; cle...",
2,3,Point 37,31.388181,74.178627,Vegetation-to-built-up,Other (no change),Missed change,New road / construction site confused with bar...,"NDVI 2019=0.57, 2025=0.29; NDBI 2025=0.05; cle...",
3,4,Point 9,31.387899,74.298212,Other (no change),Conversion detected,False conversion,Seasonal crop change,"NDVI 2019=0.39, 2025=0.25; NDBI 2025=0.09; cle...",
4,5,Point 45,31.341735,74.303916,Vegetation-to-built-up,Other (no change),Missed change,New road / construction site confused with bar...,"NDVI 2019=0.85, 2025=0.18; NDBI 2025=0.01; cle...",
5,6,Point 35,31.361507,74.227331,Vegetation (no change),Conversion detected,False conversion,Bare / fallow soil confused with built-up,"NDVI 2019=0.48, 2025=0.23; NDBI 2025=0.11; cle...",
6,7,Point 20,31.365808,74.317513,Vegetation (no change),Conversion detected,False conversion,Bare / fallow soil confused with built-up,"NDVI 2019=0.85, 2025=0.23; NDBI 2025=0.13; cle...",
7,8,Point 28,31.374730,74.251093,Vegetation-to-built-up,Vegetation (no change),Missed change,Tree-covered housing area classified as vegeta...,"NDVI 2019=0.50, 2025=0.38; NDBI 2025=-0.01; cl...",
8,9,Point 18,31.395965,74.272277,Vegetation (no change),Conversion detected,False conversion,Bare / fallow soil confused with built-up,"NDVI 2019=0.69, 2025=0.12; NDBI 2025=0.07; cle...",
9,10,Point 53,31.347996,74.261818,Vegetation-to-built-up,Other (no change),Missed change,New road / construction site confused with bar...,"NDVI 2019=0.92, 2025=0.21; NDBI 2025=0.05; cle...",


**Most common reasons:** Bare / fallow soil confused with built-up (9), New road / construction site confused with barren land (11), Seasonal crop change (2), Tree-covered housing area classified as vegetation (9), Change already present in baseline map (3), Small buildings / mixed pixels at 10 m (1), Baseline-year map error (old year mapped as built-up) (1)

**Suggested improvements:**
- Add a separate 'Construction / bare urban' class and a road-network layer (OSM).
- Add more 'Other' training samples on harvested/fallow fields and use a multi-date (e.g. Feb + Apr) composite so fields that green up at any time are recognised.
- Add texture features (e.g. GLCM on NIR) or higher-resolution imagery to detect rooftops under tree canopy.
- Improve the baseline-year map with samples drawn from baseline imagery only, and validate the baseline year separately.
- Compare images from the same crop stage, or use several dates per year (seasonal NDVI maximum) instead of a single seasonal median.
- Use higher-resolution imagery (PlanetScope / aerial) or a U-Net segmentation model trained on building footprints.
- Check the baseline year separately: add baseline-only training samples in green peri-urban plots and compare against historical high-resolution imagery.

✏️ The reasons are *suggested* from pixel evidence. Check each case against its chip (`outputs/validation/chips/point_XXX.png`) and add your own comments in the `notes` column of the validation CSV. They appear in the final tables.

## 10. All maps

In [18]:
project.make_maps()
F = project.figures
order = ["class_b", "class_r", "class_pair", "change", "change_classes", "hotspots", "dndvi", "dndbi",
         "heat", "rule_r", "samples", "val_pts", "areas", "trend", "transition", "models", "importance",
         "cv_cm", "val_cm"]
for k in order:
    if k in F:
        display(Image(filename=str(F[k]), width=620))

Output hidden; open in https://colab.research.google.com to view.

## 11. Interactive map (Folium, layer toggles) and Streamlit dashboard

In [19]:
import geoguardian.webmap as wm
wm.MAX_OVERLAY_PX = 900          # smaller overlays -> map file about 3x smaller, still sharp
project.make_web_map()
print("Interactive map saved:", project.web_map)
print("Open it from Drive: outputs/web/interactive_map.html (download, then double-click to open in Chrome)")

[webmap] saved outputs/web/interactive_map.html (26.4 MB)
Interactive map saved: outputs/web/interactive_map.html
Open it from Drive: outputs/web/interactive_map.html (download, then double-click to open in Chrome)


**Streamlit dashboard** - AOI name, year selection, RGB/NDVI/NDBI, change map, area table and observations.
Run the cell below and open the link it prints. To publish it, push the repo (including `outputs/`) to GitHub and deploy `app/streamlit_app.py` on [Streamlit Community Cloud](https://streamlit.io/cloud).

In [20]:
import subprocess, time, re, os

# --- reconnect Drive and go to the project (safe to run even if already done) ---
from google.colab import drive
drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/punjab-geoguardian"
os.chdir(PROJECT_DIR)
OUTPUTS = os.path.join(PROJECT_DIR, "outputs")
print("Results found:", os.path.exists(os.path.join(OUTPUTS, "run_summary.json")))

# --- make sure streamlit is installed (lost after a restart) ---
subprocess.run("pip install -q streamlit", shell=True)

# --- 1) start the dashboard ---
subprocess.run("pkill -f streamlit", shell=True)
env = dict(os.environ, GEOGUARDIAN_OUTPUTS=OUTPUTS)
subprocess.Popen("streamlit run app/streamlit_app.py --server.port 8501 --server.headless true "
                 "> /content/streamlit.log 2>&1", shell=True, env=env)
time.sleep(15)
code = subprocess.run("curl -s -o /dev/null -w '%{http_code}' localhost:8501",
                      shell=True, capture_output=True, text=True).stdout
print("Dashboard status:", code, "(200 = running)")

# --- 2) public link via Cloudflare tunnel ---
if not os.path.exists("/content/cloudflared"):
    subprocess.run("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/"
                   "cloudflared-linux-amd64 -O /content/cloudflared && chmod +x /content/cloudflared",
                   shell=True)
subprocess.run("pkill -f cloudflared", shell=True)
subprocess.Popen("/content/cloudflared tunnel --url http://localhost:8501 --no-autoupdate "
                 "> /content/cf.log 2>&1", shell=True)
time.sleep(12)
links = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", open("/content/cf.log").read())
print("Open your dashboard here:", links[0] if links else "no link yet - run this cell again")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Results found: True
Dashboard status: 200 (200 = running)
Open your dashboard here: https://standards-movements-acute-colour.trycloudflare.com


## 12. Exports, Week-1 summary, final report PDF and LinkedIn section

In [21]:
project.week1_summary().export().report()
display(Markdown(f"**Final report:** `{next(project.p.report.glob('*.pdf'))}`"))
display(Markdown((project.p.report / "LINKEDIN_PORTFOLIO.md").read_text()))

[export] GeoTIFFs, 3705 conversion polygons, CSV tables written
[report] saved outputs/report/Punjab_GeoGuardian_Final_Report_Hamza_Nadeem.pdf


**Final report:** `outputs/report/Punjab_GeoGuardian_Final_Report_Hamza_Nadeem.pdf`

# Punjab GeoGuardian - Detecting Urban Expansion and Farmland Loss Using Satellite Imagery

**Hamza Nadeem** | AIRI Team PITB - AI Internship Task 3

## 1. Project Overview
I built a GeoAI prototype that uses Sentinel-2 satellite imagery to detect urban expansion and
possible farmland loss in the Lahore peri-urban expansion zone near Raiwind Road, Punjab. The system compares 2019 with 2025, computes
vegetation (NDVI) and built-up (NDBI) indices, classifies every 10 m pixel as vegetation/agriculture,
built-up or other with a Random Forest model, and highlights where green or agricultural land may have
turned into built-up land. The outputs are maps, area statistics, ranked change hotspots, an
interactive layer-toggle map, a Streamlit dashboard and an automated PDF report.

## 2. Tools and Technologies Used
Python, Google Colab, Google Earth Engine, Sentinel-2 (harmonized L2A), Cloud Score+, Geemap, Folium,
NumPy, Pandas, Matplotlib, Scikit-learn (Random Forest), XGBoost, Rasterio, SciPy, Streamlit, ReportLab.

## 3. Area and Data Selection
- **AOI:** Lahore peri-urban expansion zone near Raiwind Road (225 km²) - South-western peri-urban belt of Lahore along the Raiwind Road corridor, south of Thokar Niaz Baig. The area mixes established housing, new housing schemes, roads, brick kilns and wheat/rice farmland, which makes it a typical example of Lahore's outward growth onto agricultural land.
- **Years compared:** 2019 and 2025 (plus 2022 for the trend)
- **Satellite source:** Copernicus Sentinel-2 MSI Level-2A (10 m)
- **Date ranges:** 2019: 2019-02-01 to 2019-03-31 (11 image dates); 2025: 2025-02-01 to 2025-03-31 (13 image dates) - same Feb-Mar season every year so crop stage matches
- **Cloud filter:** scenes < 20% cloud, plus pixel-level Cloud Score+ masking

## 4. Methodology
AOI selection -> Sentinel-2 loading -> date and cloud filtering -> cloud-masked median composites ->
RGB, NDVI, NDBI (+MNDWI) -> training locations (Built-up: 200, Other: 200, Vegetation: 200; minimum 50 per class), spread over the AOI and
kept separate from validation points -> Rule-based baseline vs Random Forest vs XGBoost (spatial
cross-validation) -> classification of both years -> post-classification change detection with
spectral confirmation -> area calculation -> hotspot ranking -> maps, dashboard and report.

## 5. Results
- Total AOI area: **225.00 km²**
- Vegetation / agriculture in 2019: **83.39 km²**
- Built-up area in 2025: **118.42 km²** (was 112.82 km² in 2019)
- Estimated agriculture-to-urban conversion: **12.31 km²** (1231 ha)
- Percentage of AOI converted: **5.47%**
- Main hotspots: Hotspot #1: about 46.1 ha of possible conversion (46.0% of the 1 km² cell) in the north-western part of the AOI (9.2 km from centre) [31.4627°N, 74.1864°E]. Hotspot #2: about 35.8 ha of possible conversion (35.8% of the 1 km² cell) in the north-western part of the AOI (8.5 km from centre) [31.4537°N, 74.1865°E]. Hotspot #3: about 34.3 ha of possible conversion (34.3% of the 1 km² cell) in the north-western part of the AOI (8.6 km from centre) [31.4628°N, 74.1969°E]. The top 3 one-square-kilometre cells hold 9% of all detected conversion, so the change is spread across many locations.

## 6. Validation
60 points checked, 29 correct and 31 incorrect (approximate accuracy 48.3%, area-weighted 56.6%, kappa 0.11). Common confusion areas: New road / construction site confused with barren land (11), Bare / fallow soil confused with built-up (9), Tree-covered housing area classified as vegetation (9).

## 7. Error Analysis
I analysed 36 weak or incorrect cases. The main reasons were New road / construction site confused with barren land (11), Bare / fallow soil confused with built-up (9), Tree-covered housing area classified as vegetation (9).
Dry or harvested fields can look like built-up land in the SWIR bands; tree-covered housing can look like
vegetation; and small buildings are mixed with their surroundings at 10 m resolution. Improvements:
- Add a separate 'Construction / bare urban' class and a road-network layer (OSM).
- Add more 'Other' training samples on harvested/fallow fields and use a multi-date (e.g. Feb + Apr) composite so fields that green up at any time are recognised.
- Add texture features (e.g. GLCM on NIR) or higher-resolution imagery to detect rooftops under tree canopy.

## 8. What I Learned
- How to use satellite imagery in an AI project, and how to pick an Area of Interest.
- How NDVI identifies vegetation and NDBI identifies built-up surfaces.
- How to build a land-use classifier and evaluate it honestly with spatial cross-validation.
- How to run change detection between two years and turn pixels into area estimates.
- How to validate a map with manually checked points and report its uncertainty.
- How to communicate map-based AI results for public-sector decision support.

## 9. Future Improvements
- Add more years for a full time series (annual 2019-2025).
- Run at district level with official boundaries, and compare districts.
- Use higher-resolution imagery and train a U-Net segmentation model.
- Add more land classes (water, barren, industrial/brick kilns, roads).
- Validate with official land-use data and approved scheme boundaries.
- Add automated policy-summary generation for planners.

## 10. LinkedIn Post
Today, I completed AIRI Team PITB - AI Internship Task 3, where I built an end-to-end GeoAI project
using satellite imagery and machine learning.

For this task, I developed Punjab GeoGuardian, a prototype that detects possible urban expansion and
farmland loss by comparing Sentinel-2 imagery of the Lahore peri-urban expansion zone near Raiwind Road from 2019 and 2025.
It estimates about 12.3 km² of possible agricultural-to-urban
conversion (5.5% of the study area) and ranks where it is concentrated.

The project covered the complete GeoAI workflow:
- Area of Interest selection
- Satellite imagery collection (Google Earth Engine)
- NDVI and NDBI calculation
- Land-use classification (Random Forest / XGBoost)
- Change detection
- Area estimation
- Map visualization and an interactive dashboard
- Validation and error analysis
- Final documentation

This project helped me understand how AI can be used for real-world public-sector challenges such as
urban planning, agricultural land monitoring, and environmental decision support.

Tools used: Python, Google Colab, Google Earth Engine, Sentinel-2, Geemap/Folium, Scikit-learn,
XGBoost and Streamlit.

Key learning: AI becomes more powerful when it is connected with real-world geography, public data,
and decision-making problems.

#GeoAI #RemoteSensing #SatelliteImagery #MachineLearning #UrbanPlanning #Agriculture #Python
#GoogleEarthEngine #AIRI #PITB #AIInternship

---
*Learning prototype - not an official government monitoring system. All figures are estimates of
possible / detected change, and no claim is made about any specific development.*


In [22]:
# Zip everything for submission (Drive folder already contains all files if USE_DRIVE=True)
zip_path = project.zip_outputs()
print("Submission zip:", zip_path)

Submission zip: /content/drive/MyDrive/punjab-geoguardian/punjab_geoguardian_outputs_lahore_raiwind.zip


### ✅ Submission checklist
- [ ] All 60 validation points labelled manually (no *PROVISIONAL* banner in the report)
- [ ] Error-analysis notes reviewed
- [ ] Screenshots: AOI map (part 2), interactive map (part 11), dashboard (part 11)
- [ ] Notebook saved **with outputs** (File → Save) and shared
- [ ] Repo pushed to GitHub (see README) or the Drive folder shared with the supervisor
- [ ] LinkedIn post published from `outputs/report/LINKEDIN_PORTFOLIO.md`